In [1]:
!pip install -q -U "bitsandbytes>=0.46.1"
!pip install -q -U accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 48.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 14.4 MB/s eta 0:00:00


In [2]:
import accelerate
print(f"accelerate version : {accelerate.__version__}")

accelerate version : 1.15.0


In [3]:
import transformers
import inspect
from transformers import Seq2SeqTrainer

print("Transformers:", transformers.__version__)
print("Trainer source:", inspect.getfile(Seq2SeqTrainer))

Transformers: 5.16.1
Trainer source: /usr/local/lib/python3.13/dist-packages/transformers/trainer_seq2seq.py


In [4]:
import os
import subprocess
import pandas as pd

env = os.environ.copy()
env["TORCH_DISTRIBUTED_DEBUG"] = "DETAIL"
env["CUDA_LAUNCH_BLOCKING"] = "1"
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

debug_path = "/kaggle/working/ddp_debug.log"
train_metrics_path = "/kaggle/working/training_metrics.jsonl"
eval_metrics_path = "/kaggle/working/evaluation_metrics.jsonl"

for path in (train_metrics_path, eval_metrics_path):
    with open(path, "w", encoding="utf-8"):
        pass

debug_file = open(debug_path, "w", buffering=1)

proc = subprocess.Popen(
    [
        "accelerate",
        "launch",
        "--num_processes", "2",
        "--debug",
        "/kaggle/usr/lib/notebooks/bernardchiagoziem/lb_train_script_main/notebooks/bernardchiagoziem/lb_train_script_main.py",
    ],
    stdout=debug_file,
    stderr=subprocess.STDOUT,
    env=env,
)

print(f"Training started, PID: {proc.pid}")

# return_code = proc.wait()

# print(f"Training process exited with code: {return_code}")

Training started, PID: 174


In [11]:
status = "running" if proc.poll() is None else f"exited, code={proc.returncode}"
print(f"Status: {status}")

Status: exited, code=1


In [12]:
with open(debug_path, "r", encoding="utf-8") as file:
    debug_logs = file.read()


print("\n".join(debug_logs.splitlines()[0:900]))

The following values were not passed to `accelerate launch` and had defaults used instead:
		More than one GPU was found, enabling multi-GPU training.
		If this was unintended please pass in `--num_processes=1`.
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
transformers version : 5.16.1
transformers version : 5.16.1
CUDA AVAILABLE : True
GPU 0: Tesla T4 (UUID: GPU-5072ef4b-93db-ecf2-f847-98aa40bbe080)
GPU 1: Tesla T4 (UUID: GPU-419a1322-2507-d3cd-b973-22fd9f901b23)
CUDA AVAILABLE : True
GPU 0: Tesla T4 (UUID: GPU-5072ef4b-93db-ecf2-f847-98aa40bbe080)
GPU 1: Tesla T4 (UUID: GPU-419a1322-2507-d3cd-b973-22fd9f901b23)

Loading weights: 100%|██████████| 1016/1016 [00:03<00:00, 275.15it/s]

Loading weights: 100%|██████████| 1016/1016 [00:03<00:00, 271.18it/s]
Model weight properties : 
{'infini

In [7]:
train_metrics = pd.read_json(train_metrics_path,lines=True)
display(train_metrics.head(10))

""


In [8]:
train_metrics = pd.read_json(eval_metrics_path,lines=True)
display(train_metrics.head(10))

""


In [9]:
!nvidia-smi

Thu Nov  5 17:02:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [10]:
# proc.terminate()